<a href="https://colab.research.google.com/github/tharak-bairneni/sqlite-bookstore-database-project/blob/main/SQLite_Bookstore_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Database Design & Creation

In [ ]:
import sqlite3
import pandas as pd

# Create database connection
conn = sqlite3.connect("bookstore.db")
cursor = conn.cursor()

print("✓ Database connection established")

✓ Database connection established


Create Tables

In [ ]:
# Create Books table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    price REAL NOT NULL,
    stock_quantity INTEGER DEFAULT 0
)
""")

print("✓ Books table created successfully")


# Create Customers table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    city TEXT,
    join_date TEXT
)
""")

print("✓ Customers table created successfully")


# Create Orders table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER,
    book_id INTEGER,
    quantity INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    total_amount REAL,
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id),
    FOREIGN KEY (book_id) REFERENCES Books(book_id)
)
""")

print("✓ Orders table created successfully")

conn.commit()

✓ Books table created successfully
✓ Customers table created successfully
✓ Orders table created successfully


Display Schema Using PRAGMA

In [ ]:
for table in ["Books", "Customers", "Orders"]:
    print(f"\nSchema for {table} table:")
    schema = cursor.execute(f"PRAGMA table_info({table})").fetchall()
    for row in schema:
        print(row)


Schema for Books table:
(0, 'book_id', 'INTEGER', 0, None, 1)
(1, 'title', 'TEXT', 1, None, 0)
(2, 'author', 'TEXT', 1, None, 0)
(3, 'price', 'REAL', 1, None, 0)
(4, 'stock_quantity', 'INTEGER', 0, '0', 0)

Schema for Customers table:
(0, 'customer_id', 'INTEGER', 0, None, 1)
(1, 'name', 'TEXT', 1, None, 0)
(2, 'email', 'TEXT', 1, None, 0)
(3, 'city', 'TEXT', 0, None, 0)
(4, 'join_date', 'TEXT', 0, None, 0)

Schema for Orders table:
(0, 'order_id', 'INTEGER', 0, None, 1)
(1, 'customer_id', 'INTEGER', 0, None, 0)
(2, 'book_id', 'INTEGER', 0, None, 0)
(3, 'quantity', 'INTEGER', 1, None, 0)
(4, 'order_date', 'TEXT', 1, None, 0)
(5, 'total_amount', 'REAL', 0, None, 0)


TASK 2: Data Insertion & Queries

Insert Data

In [ ]:
books_data = [
    ('Python Programming', 'John Smith', 599.99, 25),
    ('Data Science Handbook', 'Jane Doe', 899.50, 15),
    ('Machine Learning Basics', 'Alan Turing', 1299.00, 10),
    ('SQL Essentials', 'Edgar Codd', 499.99, 30),
    ('Web Development', 'Tim Berners', 799.00, 20)
]

customers_data = [
    ('Rahul Sharma', 'rahul@email.com', 'Mumbai', '2024-01-15'),
    ('Priya Patel', 'priya@email.com', 'Delhi', '2024-01-20'),
    ('Amit Kumar', 'amit@email.com', 'Bangalore', '2024-02-01'),
    ('Sneha Reddy', 'sneha@email.com', 'Hyderabad', '2024-02-10'),
    ('Vikram Singh', 'vikram@email.com', 'Mumbai', '2024-02-15')
]

orders_data = [
    (1, 1, 2, '2024-03-01', 1199.00),
    (1, 2, 1, '2024-03-02', 899.50),
    (2, 1, 1, '2024-03-03', 599.99),
    (2, 3, 1, '2024-03-05', 1299.00),
    (3, 4, 3, '2024-03-07', 1499.97),
    (4, 2, 1, '2024-03-10', 899.50),
    (5, 5, 2, '2024-03-12', 1598.00)
]

cursor.executemany("INSERT INTO Books (title, author, price, stock_quantity) VALUES (?, ?, ?, ?)", books_data)
cursor.executemany("INSERT INTO Customers (name, email, city, join_date) VALUES (?, ?, ?, ?)", customers_data)
cursor.executemany("INSERT INTO Orders (customer_id, book_id, quantity, order_date, total_amount) VALUES (?, ?, ?, ?, ?)", orders_data)

conn.commit()

print("✓ Inserted 5 books")
print("✓ Inserted 5 customers")
print("✓ Inserted 7 orders")

✓ Inserted 5 books
✓ Inserted 5 customers
✓ Inserted 7 orders


Complex Queries

In [ ]:
# Customers from Mumbai
print("Customers from Mumbai:")
print(pd.read_sql("SELECT * FROM Customers WHERE city='Mumbai'", conn))

# Books price > 800 and stock > 10
print(pd.read_sql("SELECT * FROM Books WHERE price > 800 AND stock_quantity > 10", conn))

# Total orders
print("Total Orders:",
      cursor.execute("SELECT COUNT(*) FROM Orders").fetchone()[0])

# Customer with most orders
print(pd.read_sql("""
SELECT customer_id, COUNT(*) as total_orders
FROM Orders
GROUP BY customer_id
ORDER BY total_orders DESC
LIMIT 1
""", conn))

# Total revenue
print("Total Revenue:",
      cursor.execute("SELECT SUM(total_amount) FROM Orders").fetchone()[0])

Customers from Mumbai:
   customer_id          name             email    city   join_date
0            1  Rahul Sharma   rahul@email.com  Mumbai  2024-01-15
1            5  Vikram Singh  vikram@email.com  Mumbai  2024-02-15
   book_id                  title    author  price  stock_quantity
0        2  Data Science Handbook  Jane Doe  899.5              15
Total Orders: 7
   customer_id  total_orders
0            2             2
Total Revenue: 7994.96


TASK 3: Pandas Integration

Load Tables into Pandas

In [ ]:
books_df = pd.read_sql("SELECT * FROM Books", conn)
customers_df = pd.read_sql("SELECT * FROM Customers", conn)
orders_df = pd.read_sql("SELECT * FROM Orders", conn)

books_df.head(3)
customers_df.head(3)
orders_df.head(3)

,order_id,customer_id,book_id,quantity,order_date,total_amount
0,1,1,1,2,2024-03-01,1199.00
1,2,1,2,1,2024-03-02,899.50
2,3,2,1,1,2024-03-03,599.99


Merge for Comprehensive Report

In [ ]:
report = orders_df.merge(customers_df, on="customer_id") \
                  .merge(books_df, on="book_id")

report = report[["order_id", "name", "city", "title", "quantity", "total_amount"]]

report.head()

,order_id,name,city,title,quantity,total_amount
0,1,Rahul Sharma,Mumbai,Python Programming,2,1199.00
1,2,Rahul Sharma,Mumbai,Data Science Handbook,1,899.50
2,3,Priya Patel,Delhi,Python Programming,1,599.99
3,4,Priya Patel,Delhi,Machine Learning Basics,1,1299.00
4,5,Amit Kumar,Bangalore,SQL Essentials,3,1499.97


Analysis

In [ ]:
# Average Order Value
print("Average Order Value:",
      orders_df["total_amount"].mean())

# Orders by City
print(report.groupby("city")["order_id"].count())

# Most Popular Book
print(report.groupby("title")["quantity"].sum().sort_values(ascending=False).head(1))

Average Order Value: 1142.1371428571429
city
Bangalore    1
Delhi        2
Hyderabad    1
Mumbai       3
Name: order_id, dtype: int64
title
Python Programming    3
Name: quantity, dtype: int64


Pandas to SQL (Discount Table)

In [ ]:
discounts_data = {
    'book_id': [1,2,3,4,5],
    'discount_percent': [10,15,5,20,12]
}

discounts_df = pd.DataFrame(discounts_data)

discounts_df.to_sql("discounts", conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT * FROM discounts", conn))

   book_id  discount_percent
0        1                10
1        2                15
2        3                 5
3        4                20
4        5                12


Join Books & Discounts

In [ ]:
print(pd.read_sql("""
SELECT b.title,
       b.price AS original_price,
       d.discount_percent,
       b.price * (1 - d.discount_percent/100.0) AS discounted_price
FROM Books b
JOIN discounts d
ON b.book_id = d.book_id
""", conn))

                     title  original_price  discount_percent  discounted_price
0       Python Programming          599.99                10           539.991
1    Data Science Handbook          899.50                15           764.575
2  Machine Learning Basics         1299.00                 5          1234.050
3           SQL Essentials          499.99                20           399.992
4          Web Development          799.00                12           703.120


Close Connection

In [ ]:
conn.close()
print("✓ Database connection closed")

✓ Database connection closed
